# DQN Dino AI - Google Colab Training

Train DQN on **Google Colab (GPU T4, free)**.

---
**Runtime > Change runtime type > GPU**
**Runtime > Run all**
---

## 1. Install dependencies

In [ ]:
!pip install pygame numpy torch matplotlib pillow -q

## 2. Upload project files

Drag folders `shared/`, `dqn/`, `templates/` into Colab's Files panel (left side).

Or clone from GitHub:
```
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
```

In [ ]:
import os, sys
sys.path.insert(0, '/content')
print('Content:', os.listdir('/content'))

In [ ]:
# Check structure
for p in ['shared', 'dqn', 'templates', 'shared/config.py', 'dqn/dqn_ai.py']:
    ok = os.path.exists(f'/content/{p}')
    print(f'  {"OK" if ok else "MISSING"} /content/{p}')

## 3. Headless pygame

In [ ]:
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'

import pygame
pygame.init()
pygame.display.set_mode((1, 1))
print('pygame ready')

## 4. Verify config

STATE_SIZE = 12 (2 obstacles x 5 features + speed + pad)

In [ ]:
from shared.config import STATE_SIZE, ACTION_SIZE
print(f'State: {STATE_SIZE} dims  |  Action: {ACTION_SIZE}')

## 5. Mount Google Drive

Model saves here automatically, survives Colab restart.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = '/content/drive/MyDrive/dino_dqn'
MODEL_PATH = f'{MODEL_DIR}/dqn_best.pkl'
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Model path: {MODEL_PATH}')

## 6. Training

| Episodes | Time   | Quality |
|----------|--------|---------|
| 300     | ~20min | Fair    |
| 800     | ~1-2h  | Good    |
| 2000    | ~3-4h  | Great   |

In [ ]:
import sys, time
sys.path.insert(0, '/content/dqn')
from dqn_ai import DQNDinoAI

ai = DQNDinoAI()
print(f'Network: {ai.cfg["hidden_sizes"]} | State: {ai.cfg["state_size"]}')
print(f'Device: {"CUDA" if ai.device.type == "cuda" else "CPU"}')

In [ ]:
# START TRAINING - CHANGE n_episodes HERE
start = time.time()
scores = ai.train(
    n_episodes = 800,       # CHANGE THIS
    max_steps_per_ep = 8000,
    verbose_every = 50,
    save_path = MODEL_PATH,
)
print(f'Done in {(time.time()-start)/60:.1f} min!')

## 7. Plot learning curve

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def plot(scores, save_path=None):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(scores, alpha=0.4, color='steelblue', label='Score per episode')
    if len(scores) >= 50:
        w = 50
        avg = np.convolve(scores, np.ones(w)/w, mode='valid')
        ax.plot(range(w-1, len(scores)), avg, color='orange', lw=2,
                label=f'Moving avg({w})')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Score')
    ax.set_title('DQN Training Progress')
    ax.legend()
    plt.tight_layout()
    if save_path: plt.savefig(save_path)
    plt.show()

plot(scores, f'{MODEL_DIR}/training_curve.png')

## 8. Evaluate

In [ ]:
ai.epsilon = 0.0
from shared.evaluator import evaluate

stats = evaluate(ai, n_runs=20, verbose=True)
print(f'mean={stats["mean"]:.0f} max={stats["max"]} std={stats["std"]:.0f}')

## 9. Quick test (1 episode)

In [ ]:
from shared.game_env import DinoEnv, Dinosaur

env = DinoEnv(render=False)
dino = Dinosaur(env.sprites)
state = env.reset(dino)
done = False

while not done:
    action = ai.predict(state)
    state, _, done, _ = env.step_single(dino, action)

env.close()
print(f'Episode score: {dino.score}')